In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/murtazaabdullah2010/georgian-ai-league-i-memory-trace/train_data.csv
/kaggle/input/datasets/murtazaabdullah2010/georgian-ai-league-i-memory-trace/custom_archive/model_A.joblib
/kaggle/input/datasets/murtazaabdullah2010/georgian-ai-league-i-memory-trace/custom_archive/model_B.joblib


In [14]:
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
train_ds = pd.read_csv("/kaggle/input/datasets/murtazaabdullah2010/georgian-ai-league-i-memory-trace/train_data.csv")
modelA = joblib.load("/kaggle/input/datasets/murtazaabdullah2010/georgian-ai-league-i-memory-trace/custom_archive/model_A.joblib")
modelB =joblib.load("/kaggle/input/datasets/murtazaabdullah2010/georgian-ai-league-i-memory-trace/custom_archive/model_B.joblib")

In [16]:
train_ds

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,y,row_id
0,20000,2,2,1,24,2,2,-1,-1,-2,...,482,433,123,681,801,512,436,179,1,0
1,120000,2,2,2,26,-1,2,0,0,0,...,574,429,652,645,456,826,427,856,1,1
2,90000,2,2,2,34,0,0,0,0,0,...,571,806,795,995,922,940,25,869,0,2
3,50000,2,2,1,37,0,0,0,0,0,...,618,570,476,120,6,47,450,781,0,3
4,50000,1,2,1,57,-1,0,-1,0,0,...,22,0,591,783,788,590,557,413,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,1,3,1,39,0,0,0,0,0,...,931,137,16,128,573,225,717,230,0,29995
29996,150000,1,3,2,43,-1,-1,-1,-1,0,...,571,623,526,817,874,981,200,853,0,29996
29997,30000,1,2,2,37,4,3,2,-1,0,...,640,823,685,334,568,98,593,321,1,29997
29998,80000,1,3,1,41,1,-1,0,0,0,...,620,655,315,866,149,341,952,24,1,29998


In [33]:
# Initialize empty list to store predictions
preds = []

# Get feature columns (exclude target 'y' and identifier 'row_id')
X_cols = [col for col in train_ds.columns if col not in ["y", "row_id"]]

# Iterate through each row in the dataset
for idx, row in train_ds.iterrows():
    # Extract features and reshape for model prediction (1 sample, all features)
    X = row[X_cols].values.reshape(1, -1)
    
    # Get the actual target value (0 or 1) for this row
    actual_y = row["y"]  
    
    # Get probability predictions from both models
    # predict_proba returns [[prob_class0, prob_class1]]
    probA = modelA.predict_proba(X)[0]  # [P(y=0), P(y=1)] from model A
    probB = modelB.predict_proba(X)[0]  # [P(y=0), P(y=1)] from model B
    
    # Get the probability for the actual class from each model
    # If actual_y = 0, use index 0; if actual_y = 1, use index 1
    probA_for_actual = probA[actual_y]
    probB_for_actual = probB[actual_y]
    
    # Compare which model gave higher probability to the actual class
    if probA_for_actual > probB_for_actual:
        preds.append(0)  # Model A is more confident (correctly predicted)
    else:
        preds.append(1)  # Model B is more confident (or equal, choose B)

In [34]:
sub = pd.DataFrame({
    "subtaskID":1,
    "datapointID":train_ds["row_id"],
    "answer": preds
})

In [36]:
sub.to_csv("sub2.csv", index = False)